# Lagrangian statistics and coherent structures
## 00 — Theory

This notebook is the theory of the course, with numbered equations and no code. The next notebook, [`01-lagrangian-bickley-jet.ipynb`](01-lagrangian-bickley-jet.ipynb), applies every section below, in the same order, to an analytic flow (the Bickley jet), where we know what the answer should look like. The HF-radar practical (notebook 02) then applies exactly the same functions to observed surface currents.

| Theory (this notebook) | Bickley jet ([`01-lagrangian-bickley-jet.ipynb`](01-lagrangian-bickley-jet.ipynb)) |
|---|---|
| [A1](#A1) Kinematics | [B1](01-lagrangian-bickley-jet.ipynb#B1) the flow, [B2](01-lagrangian-bickley-jet.ipynb#B2) trajectories |
| [A2](#A2) Single-particle statistics | [B3](01-lagrangian-bickley-jet.ipynb#B3) |
| [A3](#A3) Two-particle statistics | [B4](01-lagrangian-bickley-jet.ipynb#B4) |
| [A4](#A4) FSLE as a statistic | [B5](01-lagrangian-bickley-jet.ipynb#B5) |
| [A5](#A5) FTLE and LCS | [B6](01-lagrangian-bickley-jet.ipynb#B6) |
| [A6](#A6) FSLE map | [B7](01-lagrangian-bickley-jet.ipynb#B7) |
| [A7](#A7) Objectivity and LAVD | [B8](01-lagrangian-bickley-jet.ipynb#B8) |
| [A8](#A8) Eulerian diagnostics | [B9](01-lagrangian-bickley-jet.ipynb#B9) |
| [A9](#A9) Numerics | used throughout; [B2](01-lagrangian-bickley-jet.ipynb#B2) |

<a id="A0"></a>
## A0. Introduction and learning goals

An HF radar or an ocean model gives a velocity field $\mathbf u(\mathbf x,t)$ on a grid: the **Eulerian** view. Most practical questions are about where water (and what floats in it) *goes*: the **Lagrangian** view.

- **Transport and search-and-rescue:** where will an oil slick or a person in the water be in 24 h, and how fast does the search area grow?
- **Connectivity:** which spawning site feeds which nursery; how long does water stay in a bay?
- **Plastics and other floating material:** where does it accumulate?

Looking at snapshots of the velocity field does not answer these questions. In an unsteady flow, streamlines are not trajectories, and two flows with similar snapshots can mix very differently.

**Where Lagrangian data come from**

| Source | Strengths | Weaknesses |
|---|---|---|
| Real drifters | the truth, at the scales the drifter feels | few, expensive, irregular sampling, windage |
| Synthetic particles in HF-radar fields | observed currents, hourly, ~1–5 km | small footprint, gaps, measurement noise, surface only |
| Synthetic particles in model fields | full coverage, 3-D, any number of particles | model error; unresolved scales are missing |

**After this notebook and the next one you should be able to**

1. integrate particle trajectories and know when the result can be trusted;
2. compute and interpret absolute dispersion, Lagrangian time scales and diffusivity;
3. compute relative dispersion and the FSLE, and relate them to dispersion regimes;
4. compute FTLE and FSLE maps and read their ridges as transport barriers;
5. detect coherent eddies with the LAVD and explain why this is preferable to Okubo–Weiss;
6. list the numerical choices that can fake or destroy each of these signals.

**Roadmap: which diagnostic answers which question**

| Diagnostic | Type | Question it answers |
|---|---|---|
| Absolute dispersion, $T_L$, $K$ | one-particle statistic | How far does a particle wander from its release point, and can that be modelled as diffusion? |
| Relative dispersion, kurtosis | two-particle statistic | How fast does a cloud spread about its own centre, as a function of **time**? |
| FSLE $\lambda(\delta)$ | two-particle statistic | How fast do pairs separate, as a function of **scale**? |
| FTLE map | map | Where are the strongest repelling / attracting material lines over a time $T$? |
| FSLE map | map | Same question, fixing the separation scale instead of the time |
| LAVD | map | Which water masses rotate coherently and keep their identity? |
| Okubo–Weiss | Eulerian map | Where does rotation dominate strain *right now*? (comparison only) |

<a id="A1"></a>
## A1. Kinematics

**Eulerian and Lagrangian descriptions.** A property $c$ measured at a fixed point changes at the rate $\partial_t c$. Following a water parcel it changes at the rate given by the **material derivative**

$$\frac{D}{Dt}=\frac{\partial}{\partial t}+\mathbf u\cdot\nabla . \tag{A1.1}$$

**Trajectories.** A parcel released at $\mathbf x_0$ at time $t_0$ follows

$$\frac{d\mathbf x}{dt}=\mathbf u(\mathbf x,t),\qquad \mathbf x(t_0)=\mathbf x_0 . \tag{A1.2}$$

Everything in this course is a statistic or a map built from solutions of (A1.2).

**Flow map.** Collecting the solutions for all initial positions defines the flow map

$$F_{t_0}^{t}:\ \mathbf x_0\ \mapsto\ \mathbf x(t;t_0,\mathbf x_0). \tag{A1.3}$$

Its Jacobian $\nabla F_{t_0}^{t}$ is the **deformation gradient**. It tells how a small initial separation $\boldsymbol\xi_0$ between two parcels has changed: $\boldsymbol\xi(t)\approx\nabla F_{t_0}^{t}(\mathbf x_0)\,\boldsymbol\xi_0$. It obeys

$$\frac{d}{dt}\nabla F_{t_0}^{t}=\nabla\mathbf u\big(\mathbf x(t),t\big)\,\nabla F_{t_0}^{t},\qquad \nabla F_{t_0}^{t_0}=\mathbf I . \tag{A1.4}$$

**Area change.** From (A1.4),

$$\det\nabla F_{t_0}^{t}=\exp\!\left(\int_{t_0}^{t}\nabla\!\cdot\mathbf u\,\big(\mathbf x(s),s\big)\,ds\right). \tag{A1.5}$$

For a non-divergent 2-D flow, $\det\nabla F=1$: material patches change shape but keep their area. This is the case for the Bickley jet of the next notebook.

**Surface currents are divergent.** The horizontal velocity at the sea surface (HF radar, or the top layer of a model) is *not* non-divergent, because water upwells and downwells. Consequences to keep in mind in the HF-radar practical (notebook 02):

- floating particles accumulate in convergence zones and leave divergence zones, so a uniform release does not stay uniform;
- patches change area, so $\det\nabla F\neq1$ and the two eigenvalues of the Cauchy–Green tensor ([A5](#A5)) are no longer reciprocal;
- forward and backward diagnostics are not mirror images of each other;
- finite differences of noisy radar velocities add spurious divergence on top of the real one.

<a id="A2"></a>
## A2. Single-particle statistics

Release many particles and follow each one. Averages $\langle\cdot\rangle$ are over particles.

**Absolute dispersion** is the variance of the displacement about the mean displacement:

$$A^2(t)=\Big\langle\big[\mathbf x(t)-\mathbf x_0\big]^2\Big\rangle . \tag{A2.1}$$

It is usually computed per component, $A_x^2$ and $A_y^2$, because ocean flows are rarely isotropic.

**Lagrangian velocity autocorrelation.** With $u'$ the velocity along a trajectory minus its mean,

$$R_L(\tau)=\frac{\langle u'(t)\,u'(t+\tau)\rangle}{\langle u'^2\rangle}. \tag{A2.2}$$

$R_L$ measures how long a particle "remembers" its velocity. The **Lagrangian integral time scale** is

$$T_L=\int_0^{\infty}R_L(\tau)\,d\tau . \tag{A2.3}$$

In practice $R_L$ is noisy at long lags and often oscillates, so the integral is cut off. The convention used here is to integrate up to the **first zero crossing** of $R_L$. Other conventions (integrate to the maximum of the running integral, or fit an exponential) give different numbers, so always state which one was used.

**Taylor's theorem (1921).** For stationary, homogeneous turbulence,

$$A^2(t)=2\langle u'^2\rangle\int_0^{t}(t-\tau)\,R_L(\tau)\,d\tau , \tag{A2.4}$$

with two limits:

$$t\ll T_L:\quad A^2\simeq\langle u'^2\rangle\,t^2\qquad\text{(ballistic)}, \tag{A2.5}$$

$$t\gg T_L:\quad A^2\simeq 2Kt,\qquad K=\langle u'^2\rangle\,T_L\qquad\text{(diffusive)}. \tag{A2.6}$$

$K$ is the single-particle (eddy) diffusivity, the number a coarse model needs to represent unresolved stirring.

**Assumptions, and what breaks them**

- *Stationarity and homogeneity.* (A2.4) assumes the statistics of $u'$ are the same everywhere and at all times.
- *Mean-flow removal.* If particles sample different mean velocities (a sheared jet), the difference in mean velocity is counted as "$u'$". $R_L$ then never decays, $T_L$ does not converge and $A^2$ keeps growing like $t^2$. This is **shear dispersion**, not diffusion.
- *Bounded domains.* $A^2$ cannot exceed the size of the region the particles can reach; it saturates.
- *Practical alternative.* Bin velocities and displacements geographically, remove the local mean in each bin, and compute a **pseudo-Eulerian** diffusivity per bin. This is the standard treatment of real drifter data.

<a id="A3"></a>
## A3. Two-particle statistics

Single-particle statistics describe the wandering of a cloud's centre **and** its spreading together. To isolate the spreading, follow **pairs** of particles.

For a pair with separation $\mathbf D=\mathbf x_1-\mathbf x_2$ and $D=|\mathbf D|$:

$$\text{relative dispersion:}\qquad \langle D^2\rangle(t), \tag{A3.1}$$

$$\text{relative diffusivity:}\qquad K_{rel}=\frac12\frac{d\langle D^2\rangle}{dt}, \tag{A3.2}$$

$$\text{kurtosis:}\qquad ku=\frac{\langle D^4\rangle}{\langle D^2\rangle^2}. \tag{A3.3}$$

The kurtosis measures how different the pairs are from each other. A large value means a few pairs have separated a lot while most are still close.

**Link with the energy spectrum.** Two particles are pulled apart by eddies. Which eddies dominate depends on the slope of the kinetic-energy spectrum $E(k)\propto k^{-\beta}$ (Bennett 1984; Babiano et al. 1990; LaCasce 2008):

| Regime | Condition | $\langle D^2\rangle$ | $ku$ |
|---|---|---|---|
| **Non-local** (exponential) | $\beta\ge3$: separation controlled by eddies larger than $D$ | $\propto e^{2\lambda t}$ | grows rapidly |
| **Local** | $1<\beta<3$: controlled by eddies of size $\sim D$ | $\propto t^{4/(3-\beta)}$ | constant |
| Richardson | $\beta=5/3$ | $\propto t^{3}$ | 5.6 |
| **Diffusive** | $D$ larger than the largest eddies: uncorrelated velocities | $\simeq4Kt$ | 2 |
| **Saturation** | bounded domain | constant | — |

In the diffusive limit the two particles move independently, so the relative diffusivity is twice the single-particle one: $\langle D^2\rangle\simeq4Kt$.

**Caveat.** These are asymptotic laws. At a *fixed time* the pairs are spread over many scales, so several regimes are mixed in one average. A flow that goes from exponential growth to saturation necessarily passes through intermediate slopes on a log-log plot, and a segment of it can look like $t^3$ over half a decade without any Richardson physics. Never identify a regime from $\langle D^2\rangle(t)$ alone; check the kurtosis and use a fixed-scale statistic ([A4](#A4)).

<a id="A4"></a>
## A4. The finite-size Lyapunov exponent as a statistic

Instead of averaging separations at fixed **time**, average times at fixed **scale** (Aurell et al. 1997; Artale et al. 1997).

Choose thresholds growing by a constant factor $r$ (here $r=\sqrt2$):

$$\delta_n=\delta_0\,r^{\,n},\qquad n=0,1,2,\dots \tag{A4.1}$$

For each pair, measure the time $\tau_n$ it takes to grow from $\delta_n$ to $\delta_{n+1}$ (first crossing to first crossing). The FSLE is

$$\lambda(\delta_n)=\frac{\ln r}{\langle\tau_n\rangle}. \tag{A4.2}$$

**Expected scalings**

| Regime | $\lambda(\delta)$ |
|---|---|
| Exponential (non-local) | constant: a plateau equal to the growth rate |
| Local, $E\propto k^{-\beta}$ | $\propto\delta^{(\beta-3)/2}$; Richardson: $\delta^{-2/3}$ |
| Ballistic / pure shear | $\propto\delta^{-1}$ |
| Diffusive | $\propto\delta^{-2}$ |

Because each scale is treated separately, the regimes that are mixed in $\langle D^2\rangle(t)$ are separated in $\lambda(\delta)$.

**Survivorship bias.** Only pairs that actually reach $\delta_{n+1}$ before the end of the record contribute to $\langle\tau_n\rangle$. Pairs trapped together in an eddy never do. At large $\delta$ the average is therefore over the fast pairs only, and $\lambda$ is biased high. Always plot the **number of pairs per bin** next to $\lambda(\delta)$.

**Why $\lambda_{FSLE}$ differs from the rate fitted to $\langle D^2\rangle$.** Suppose every pair grows exponentially but each with its own rate, $D_i=\delta_0e^{\lambda_i t}$. Then

$$\langle D^2\rangle=\delta_0^2\,\big\langle e^{2\lambda_i t}\big\rangle \tag{A4.3}$$

is dominated more and more by the **fastest** pairs: the rate fitted to $\ln\langle D^2\rangle$ tends to the largest $\lambda_i$, and the kurtosis explodes. The FSLE averages *times*, to which slow pairs contribute most, so it gives a typical rate. The same is true of $d\langle\ln D\rangle/dt$. When the two rates differ strongly, the kurtosis is high, and that is a signature of a flow with coexisting regions of fast and slow stretching.

<a id="A5"></a>
## A5. FTLE and Lagrangian coherent structures

Now ask the stretching question **pointwise**: for a parcel starting at $\mathbf x_0$, how much has the surrounding fluid been stretched after a time $T$?

**Cauchy–Green tensor.** An initial separation $\boldsymbol\xi_0$ becomes $\nabla F\,\boldsymbol\xi_0$, with squared length $\boldsymbol\xi_0^\top C\,\boldsymbol\xi_0$, where

$$C_{t_0}^{T}(\mathbf x_0)=\big(\nabla F_{t_0}^{t_0+T}\big)^{\!\top}\,\nabla F_{t_0}^{t_0+T} \tag{A5.1}$$

is the right Cauchy–Green strain tensor. It is symmetric and positive definite, with eigenvalues $0<\lambda_1\le\lambda_2$. In 2-D the largest one has the closed form

$$\lambda_2=\frac{C_{11}+C_{22}}{2}+\sqrt{\Big(\frac{C_{11}-C_{22}}{2}\Big)^{2}+C_{12}^{2}} . \tag{A5.2}$$

For a non-divergent flow $\lambda_1\lambda_2=\det C=1$: stretching by $\sqrt{\lambda_2}$ in one direction comes with compression by the same factor in the other.

**Finite-time Lyapunov exponent.** The largest possible stretching factor is $\sqrt{\lambda_2}$; expressed as an exponential rate,

$$\sigma_{t_0}^{T}(\mathbf x_0)=\frac{1}{2|T|}\,\ln\lambda_2\big(C_{t_0}^{T}(\mathbf x_0)\big). \tag{A5.3}$$

**Ridges and LCS.** Particles on either side of a *repelling* material line separate fast, so the **forward** FTLE ($T>0$) is high along it. Integrating backward ($T<0$), particles separate fast across lines that were *attracting* in forward time. Therefore (Shadden et al. 2005):

- ridges of forward FTLE $\approx$ **repelling LCS** (stable manifolds of hyperbolic trajectories);
- ridges of backward FTLE $\approx$ **attracting LCS** (unstable manifolds). These are the lines along which tracers such as chlorophyll, oil or debris align.

Where a repelling and an attracting LCS cross there is a **hyperbolic point**. The two families intersect repeatedly and enclose *lobes*; exchange of water between regions happens by lobes being carried across, not by diffusion. Regions with low FTLE in both directions (eddy cores, jet cores) are where water stays together.

**Limits in $T$**

- $T\to0$: $\sigma\to$ the largest eigenvalue of the strain-rate tensor $\tfrac12(\nabla\mathbf u+\nabla\mathbf u^\top)$, a purely Eulerian quantity.
- $T\to\infty$ (when the limit exists): $\sigma\to$ the classical Lyapunov exponent, the same for almost all points of a chaotic region. The map then loses its contrast.
- In between, $T$ should be comparable to the time scale of the transport problem at hand.

**A warning.** FTLE ridges are a heuristic. High FTLE can also be produced by shear (e.g. the flanks of a jet) without any hyperbolic structure, and ridges need not be exactly material. Rigorous definitions of LCS exist (variational theory; Haller 2015) and are compared in Hadjighasem et al. (2017). We do not need them here, but you should know that they exist.

<a id="A6"></a>
## A6. FSLE map

The FSLE of [A4](#A4) can also be mapped (d'Ovidio et al. 2004). Around each grid point $\mathbf x_0$ place neighbours at a distance $\delta_0$, integrate, and record the first time $\tau$ at which the separation reaches a prescribed $\delta_f$:

$$\lambda(\mathbf x_0;\delta_0,\delta_f)=\frac{1}{\tau}\ln\frac{\delta_f}{\delta_0}. \tag{A6.1}$$

Differences from the FTLE:

| | FTLE | FSLE map |
|---|---|---|
| Fixed | integration time $T$ | separations $\delta_0$, $\delta_f$ |
| Measured | stretching after $T$ | time to reach $\delta_f$ |
| Points that never separate | small positive value | $\lambda=0$ by convention |
| Sensitivity | linearised (infinitesimal) | finite: feels structures up to size $\delta_f$ |

Practical points:

- If $\delta_f$ is not reached within the maximum integration time $T_{max}$, set $\lambda=0$. The smallest non-zero value the map can show is therefore $\ln(\delta_f/\delta_0)/T_{max}$.
- $\tau$ is known to within one time step (or one output interval), so large $\lambda$ values are **quantised**. Interpolating the crossing time within the step reduces this.
- Choose $\delta_0$ close to the grid spacing of the map and $\delta_f$ close to the size of the structures that organise the transport (eddies, fronts). A larger $\delta_f$ keeps only the strongest lines.
- As for FTLE, backward integration gives attracting lines.

<a id="A7"></a>
## A7. Objectivity and the LAVD

**Objectivity.** Two observers, one on the shore and one on a ship that translates and turns, must agree on which water belongs to an eddy. A diagnostic is **objective** (frame-indifferent) if it is unchanged under any change of observer

$$\mathbf x=Q(t)\,\mathbf y+\mathbf b(t),\qquad Q(t)\ \text{a rotation}. \tag{A7.1}$$

Material quantities such as trajectories, separations and FTLE are objective. Many Eulerian quantities are not:

- **vorticity** changes by a constant in a rotating frame: $\tilde\omega=\omega-2\dot\theta(t)$, where $\dot\theta$ is the rotation rate of the frame;
- the **Okubo–Weiss** parameter ([A8](#A8)) contains $\omega^2$, so it changes too. An "eddy" defined by $W<0$ can appear or vanish when the observer changes.

**LAVD.** The trick of Haller et al. (2016) is to notice that the *deviation* of the vorticity from its spatial mean $\bar\omega(t)$ is objective, since the frame term cancels. Integrating its magnitude along a trajectory gives the **Lagrangian-averaged vorticity deviation**

$$\mathrm{LAVD}_{t_0}^{t_0+T}(\mathbf x_0)=\int_{t_0}^{t_0+T}\Big|\,\omega\big(\mathbf x(s),s\big)-\bar\omega(s)\Big|\,ds . \tag{A7.2}$$

It is twice the total angle by which a small fluid element at $\mathbf x_0$ has rotated, relative to the mean rotation of the fluid, over the interval.

**Rotationally coherent eddies.** In a coherent eddy all the fluid elements of a material ring rotate by the same amount, so the ring does not filament. This leads to the definition:

- an **eddy centre** is a local maximum of the LAVD field;
- the **eddy boundary** is the outermost closed LAVD contour around that maximum that is (nearly) convex.

"Nearly convex" is measured by the **convexity deficiency**

$$\mathrm{CD}=\frac{A_{hull}-A}{A}\ \le\ \mathrm{CD}_{max}, \tag{A7.3}$$

where $A$ is the area enclosed by the contour and $A_{hull}$ the area of its convex hull. Typical $\mathrm{CD}_{max}$: $10^{-3}$–$10^{-2}$. Minimum-size and minimum-peak filters remove noise. The result depends on $T$: an eddy is coherent *over a given time interval*.

<a id="A8"></a>
## A8. Eulerian diagnostics for comparison: Okubo–Weiss

The classical way to find eddies in a velocity snapshot uses the **Okubo–Weiss** parameter (Okubo 1970; Weiss 1991)

$$W=s_n^2+s_s^2-\omega^2, \tag{A8.1}$$

$$s_n=\frac{\partial u}{\partial x}-\frac{\partial v}{\partial y},\qquad s_s=\frac{\partial v}{\partial x}+\frac{\partial u}{\partial y},\qquad \omega=\frac{\partial v}{\partial x}-\frac{\partial u}{\partial y}. \tag{A8.2}$$

For a non-divergent flow the eigenvalues of $\nabla\mathbf u$ are $\pm\tfrac12\sqrt{W}$. Where $W<0$ they are imaginary (rotation dominates, "elliptic"); where $W>0$ they are real (strain dominates, "hyperbolic"). The usual rule marks eddies as

$$W<-0.2\,\sigma_W , \tag{A8.3}$$

with $\sigma_W$ the spatial standard deviation of $W$.

**Assumptions and failure modes**

- It assumes the velocity gradient changes slowly *following a parcel*. This fails near eddy edges and in unsteady flows, exactly where the boundary must be decided.
- It is instantaneous: it says nothing about whether the water inside stays inside.
- It is not objective ([A7](#A7)).
- The threshold 0.2 is arbitrary, and $\sigma_W$ depends on the domain.
- $W$ is built from products of velocity derivatives, so noise in radar or altimetry data is strongly amplified.

It is cheap and useful for a first look, which is why we compare it with the LAVD in [B9](01-lagrangian-bickley-jet.ipynb#B9).

<a id="A9"></a>
## A9. Numerics

Every diagnostic above is only as good as the trajectories behind it.

**Time integration.** We use the classical 4th-order Runge–Kutta scheme. For a step $h$ (negative for backward integration):

$$\begin{aligned}
\mathbf k_1&=\mathbf u(\mathbf x_n,t_n), &
\mathbf k_2&=\mathbf u(\mathbf x_n+\tfrac h2\mathbf k_1,\ t_n+\tfrac h2),\\
\mathbf k_3&=\mathbf u(\mathbf x_n+\tfrac h2\mathbf k_2,\ t_n+\tfrac h2), &
\mathbf k_4&=\mathbf u(\mathbf x_n+h\,\mathbf k_3,\ t_n+h),
\end{aligned}$$

$$\mathbf x_{n+1}=\mathbf x_n+\frac h6\big(\mathbf k_1+2\mathbf k_2+2\mathbf k_3+\mathbf k_4\big). \tag{A9.1}$$

The global error is $O(h^4)$. Choose $h$ so that a particle moves a small fraction of the smallest flow scale per step, $|\mathbf u|\,h\ll L_{min}$, and **check by halving $h$**. In a chaotic region individual trajectories will still differ after a long time (errors grow like $e^{\sigma t}$), but statistics and maps should not.

**Interpolation.** With gridded data the velocity must be interpolated in space and time. Linear interpolation is robust and does not overshoot, but it makes the velocity gradient piecewise constant. Below the grid spacing, particle separation reflects the interpolation, not the ocean. With hourly radar data and strong tides, linear interpolation in time also damps the tidal ellipse slightly.

**Flow-map gradients.** $\nabla F$ in (A5.1) is computed by finite differences between neighbouring particles of the initial grid, spacing $\Delta x$. Two neighbours on opposite sides of a repelling line end up $\Delta x\,e^{\sigma T}$ apart. Once this exceeds the size of the flow structures, the finite difference no longer approximates a derivative: the FTLE value saturates and thin ridges break into dots (**aliasing**). Long integration times need finer grids. A rule of thumb is $\Delta x\,e^{\sigma_{max}T}\lesssim L_{structure}$.

**Boundaries and exit times.** Real fields end at coasts, at the edge of radar coverage and in data gaps. A particle that reaches such a place must be **stopped and flagged**, not extrapolated. Each particle then has an *exit time*; statistics must be computed over the survivors (and that number reported), and map values must be masked where the particles left too early.

**Next:** [`01-lagrangian-bickley-jet.ipynb`](01-lagrangian-bickley-jet.ipynb) applies each of these sections to the Bickley jet.

## References

- Artale, V., Boffetta, G., Celani, A., Cencini, M., & Vulpiani, A. (1997). Dispersion of passive tracers in closed basins: Beyond the diffusion coefficient. *Physics of Fluids*, 9(11), 3162–3171. https://doi.org/10.1063/1.869433
- Aurell, E., Boffetta, G., Crisanti, A., Paladin, G., & Vulpiani, A. (1997). Predictability in the large: an extension of the concept of Lyapunov exponent. *Journal of Physics A: Mathematical and General*, 30(1), 1–26. https://doi.org/10.1088/0305-4470/30/1/003
- Babiano, A., Basdevant, C., Le Roy, P., & Sadourny, R. (1990). Relative dispersion in two-dimensional turbulence. *Journal of Fluid Mechanics*, 214, 535–557. https://doi.org/10.1017/S0022112090000258
- Bennett, A. F. (1984). Relative dispersion: local and nonlocal dynamics. *Journal of the Atmospheric Sciences*, 41(11), 1881–1886. https://doi.org/10.1175/1520-0469(1984)041<1881:RDLAND>2.0.CO;2
- d'Ovidio, F., Fernández, V., Hernández-García, E., & López, C. (2004). Mixing structures in the Mediterranean Sea from finite-size Lyapunov exponents. *Geophysical Research Letters*, 31(17), L17203. https://doi.org/10.1029/2004GL020328
- Hadjighasem, A., Farazmand, M., Blazevski, D., Froyland, G., & Haller, G. (2017). A critical comparison of Lagrangian methods for coherent structure detection. *Chaos*, 27(5), 053104. https://doi.org/10.1063/1.4982720
- Haller, G. (2015). Lagrangian coherent structures. *Annual Review of Fluid Mechanics*, 47, 137–162. https://doi.org/10.1146/annurev-fluid-010313-141322
- Haller, G., Hadjighasem, A., Farazmand, M., & Huhn, F. (2016). Defining coherent vortices objectively from the vorticity. *Journal of Fluid Mechanics*, 795, 136–173. https://doi.org/10.1017/jfm.2016.151
- LaCasce, J. H. (2008). Statistics from Lagrangian observations. *Progress in Oceanography*, 77(1), 1–29. https://doi.org/10.1016/j.pocean.2008.02.002
- Okubo, A. (1970). Horizontal dispersion of floatable particles in the vicinity of velocity singularities such as convergences. *Deep Sea Research and Oceanographic Abstracts*, 17(3), 445–454. https://doi.org/10.1016/0011-7471(70)90059-8
- Rypina, I. I., Brown, M. G., Beron-Vera, F. J., Koçak, H., Olascoaga, M. J., & Udovydchenkov, I. A. (2007). On the Lagrangian dynamics of atmospheric zonal jets and the permeability of the stratospheric polar vortex. *Journal of the Atmospheric Sciences*, 64(10), 3595–3610. https://doi.org/10.1175/JAS4036.1
- Shadden, S. C., Lekien, F., & Marsden, J. E. (2005). Definition and properties of Lagrangian coherent structures from finite-time Lyapunov exponents in two-dimensional aperiodic flows. *Physica D*, 212(3–4), 271–304. https://doi.org/10.1016/j.physd.2005.10.007
- Taylor, G. I. (1921). Diffusion by continuous movements. *Proceedings of the London Mathematical Society*, s2-20(1), 196–212 (issue dated 1922). https://doi.org/10.1112/plms/s2-20.1.196
- Weiss, J. (1991). The dynamics of enstrophy transfer in two-dimensional hydrodynamics. *Physica D*, 48(2–3), 273–294. https://doi.org/10.1016/0167-2789(91)90088-Q